# 💰 Phase 3b: Salary Forecasting
Regression — Predict Expected Salary (LPA)

In [ ]:
import pandas as pd, numpy as np, warnings, pickle, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

X_train = pd.read_csv('data/X_train_salary.csv')
X_test  = pd.read_csv('data/X_test_salary.csv')
y_train = pd.read_csv('data/y_train_salary.csv').squeeze()
y_test  = pd.read_csv('data/y_test_salary.csv').squeeze()
print("✅ Data loaded | Train:", X_train.shape, "| Target range:", f"{y_train.min():.1f} - {y_train.max():.1f} LPA")


In [ ]:
models = {
    'Linear Regression':    LinearRegression(),
    'Ridge Regression':     Ridge(alpha=1.0),
    'Random Forest':        RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':              XGBRegressor(n_estimators=200, random_state=42, verbosity=0),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2  = r2_score(y_test, y_pred)
    results[name] = {'model':model, 'pred':y_pred, 'mae':mae, 'rmse':rmse, 'r2':r2}
    print(f"{name:22s} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.4f}")

best_name = max(results, key=lambda k: results[k]['r2'])
print(f"\n🏆 Best Model: {best_name}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Actual vs Predicted scatter
y_pred_best = results[best_name]['pred']
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='#3498db', s=15)
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn,mx],[mn,mx], 'r--', linewidth=2, label='Perfect')
axes[0].set_title(f'Actual vs Predicted\n{best_name}', fontweight='bold')
axes[0].set_xlabel('Actual Salary (LPA)'); axes[0].set_ylabel('Predicted Salary (LPA)')
axes[0].legend()

# Residuals
residuals = y_test - y_pred_best
axes[1].hist(residuals, bins=40, color='#9b59b6', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted)'); axes[1].set_ylabel('Count')

# R² comparison
names = list(results.keys())
r2s   = [results[n]['r2'] for n in names]
colors = ['#2ecc71' if n==best_name else '#3498db' for n in names]
bars = axes[2].bar(names, r2s, color=colors, edgecolor='white')
for bar, v in zip(bars, r2s):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[2].set_ylim(0, 1.05)
axes[2].set_title('R² Score Comparison', fontweight='bold')
axes[2].set_ylabel('R² Score')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/09_salary_model.png', bbox_inches='tight')
plt.show()


In [ ]:
os.makedirs('models', exist_ok=True)
with open('models/salary_model.pkl','wb') as f:
    pickle.dump(results[best_name]['model'], f)
print(f"✅ Saved models/salary_model.pkl  ({best_name})")
print(f"   MAE: {results[best_name]['mae']:.2f} LPA | R²: {results[best_name]['r2']:.4f}")
